In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
from model_utils.plots import plot_results
import time
import itertools

In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, num_layers=1, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, 
                           num_layers=num_layers, batch_first=True, dropout=dropout)
        self.linear = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        predictions = self.linear(lstm_out[:, -1, :])
        return predictions

In [3]:
class TimeSeriesDataset(Dataset):
    def __init__(self, target_data, seq_length):
        self.target_data = target_data
        self.seq_length = seq_length
    
    def __len__(self):
        return len(self.target_data) - self.seq_length
    
    def __getitem__(self, idx):
        target_seq = self.target_data[idx:idx+self.seq_length]
        y = self.target_data[idx+self.seq_length]
        
        x = target_seq.reshape(-1, 1)
        
        return torch.FloatTensor(x), torch.FloatTensor([y])

In [ ]:
def lstm_experiment(df, target, train_size=500, val_size=100, forecast_window=161, 
                   seq_length=30, epochs=100, batch_size=32, lr=0.001, dropout=0.0, hidden_size=32, num_layers=1, plot=False, patience=100):

    total_train_val = train_size + val_size
    train = df[target][-(total_train_val+forecast_window):-(val_size+forecast_window)].values
    val = df[target][-(val_size+forecast_window):-forecast_window].values
    test = df[target][-forecast_window:].values
    
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()

    input_size = 1
    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    print(f"Training LSTM on {device_name} with hidden_size={hidden_size}, batch_size={batch_size}, num_layers={num_layers}, dropout={dropout}")
    
    # Create Datasets
    train_dataset = TimeSeriesDataset(train_scaled, seq_length)
    
    use_pin_memory = torch.cuda.is_available()
    
    # Reduced batch size for more updates per epoch (adjust as needed)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=use_pin_memory)
    
    val_dataset = TimeSeriesDataset(val_scaled, seq_length)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, pin_memory=use_pin_memory)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Scheduler to reduce LR when validation loss plateaus
    #scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    train_losses = []
    val_losses = []
    
    best_val_loss = float('inf')
    best_model_path = 'best_lstm_model.pth'
    epochs_no_improve = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            # Reshape input to (batch_size, seq_len, input_size) if needed by dataset or model
            batch_x = batch_x.view(-1, seq_length, 1)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_train_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                batch_x = batch_x.view(-1, seq_length, 1) # Ensure shape
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0
        val_losses.append(avg_val_loss)
        
        # Save best model and Early Stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        #if epochs_no_improve >= patience:
        #    print(f"Early stopping triggered at epoch {epoch+1}")
        #    break
        
        # Step the scheduler
        #scheduler.step(avg_val_loss)
        
        if plot and (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
    
    # Load the best model
    model.load_state_dict(torch.load(best_model_path))
    print(f"Loaded best model with val loss: {best_val_loss:.6f}")

    model.eval()
    forecast = []
    
    current_seq = val_scaled[-seq_length:].tolist()
    
    with torch.no_grad():
        for step in range(forecast_window):
            # Ensure input shape matches (1, seq_length, 1) for single prediction
            x = np.array(current_seq[-seq_length:]).reshape(-1, seq_length, 1) 
            x = torch.FloatTensor(x).to(device)
            
            pred = model(x).cpu().numpy()[0, 0]
            forecast.append(pred)
            
            current_seq.append(pred) # Append new prediction
    
    forecast = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    
    if plot:
        train_index = df[target][-(total_train_val+forecast_window):-(val_size+forecast_window)].index
        val_index = df[target][-(val_size+forecast_window):-forecast_window].index
        test_index = df[target][-forecast_window:].index

        plot_results(train, val, test, forecast, train_index, val_index, test_index, train_losses, val_losses, target)
    
    return model, scaler, forecast, rmse, mae, test

In [5]:
ITEM_ID = 26008       # Example item_id from sample_head.csv
STORE_ID = 6269        # Example store_id from sample_head.csv
DATA_PATH = '../dataset/data_andre.feather' # Adjust path if needed
TARGET_COL = 'value'
DATE_COL = 'date'

# To converge in fewer epochs (even if each epoch takes longer), we:
# 1. Decrease Batch Size -> More updates per epoch
# 2. Can increase Hidden Size slightly -> More capacity
# 3. Increase learning rate slightly but use scheduler (already added)

EPOCHS = 1000            # Set a bit higher to ensure convergence, though fewer should be needed
BATCH_SIZE = 4         # Smaller batch size for more frequent updates
LEARNING_RATE = 0.001  # Standard LR
HIDDEN_SIZE = 64      # Increased capacity

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


In [6]:
# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
DATA_PATH = '../dataset/subset_set.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df_full = pd.read_feather(DATA_PATH)

print(f"Filtering for Item: {ITEM_ID}, Store: {STORE_ID}...")
# Filter for specific item and store
df = df_full[(df_full['item_id'] == ITEM_ID) & (df_full['store_id'] == STORE_ID)].copy()

# Reset index to move 'date' from index to a column
df = df.reset_index()

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]

print(len(y))


# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
# Define split sizes
train_size = 500
val_size = 100
forecast_horizon = 161

lookback_window = 30

Loading data from ../dataset/subset_set.feather...
Filtering for Item: 26008, Store: 6269...
761


In [7]:
'''
print(f"Training LSTM with train_size={train_size}, val_size={val_size}, forecast_horizon={forecast_horizon}, lookback_window={lookback_window}...")

# Increased model complexity (num_layers=2) to help learn faster per epoch, even if epoch is slower
model, scaler, forecast, rmse, mae, test = lstm_experiment(
    df=df, 
    target=TARGET_COL, 
    train_size=train_size, 
    val_size=val_size,
    forecast_window=forecast_horizon, 
    seq_length=lookback_window,
    hidden_size=HIDDEN_SIZE, 
    num_layers=1,           # Increased number of layers
    dropout=0.0,            # Added dropout for regularization with more layers
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    lr=LEARNING_RATE, 
    plot=True
)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
'''

'\nprint(f"Training LSTM with train_size={train_size}, val_size={val_size}, forecast_horizon={forecast_horizon}, lookback_window={lookback_window}...")\n\n# Increased model complexity (num_layers=2) to help learn faster per epoch, even if epoch is slower\nmodel, scaler, forecast, rmse, mae, test = lstm_experiment(\n    df=df, \n    target=TARGET_COL, \n    train_size=train_size, \n    val_size=val_size,\n    forecast_window=forecast_horizon, \n    seq_length=lookback_window,\n    hidden_size=HIDDEN_SIZE, \n    num_layers=1,           # Increased number of layers\n    dropout=0.0,            # Added dropout for regularization with more layers\n    epochs=EPOCHS, \n    batch_size=BATCH_SIZE, \n    lr=LEARNING_RATE, \n    plot=True\n)\n\nprint(f"RMSE: {rmse:.4f}")\nprint(f"MAE: {mae:.4f}")\n'

# Grid Search Cell

In [ ]:
import time
import itertools
import sys
import os
sys.path.append(os.path.abspath('..'))
from model_utils.plots import plot_results

    
# Modify lstm_experiment to return timings and handle plotting internally
def lstm_experiment_grid(df, target, train_size=500, val_size=100, forecast_window=161, 
                   seq_length=30, epochs=100, batch_size=32, lr=0.001, dropout=0.0, hidden_size=32, num_layers=1, 
                   patience=50, seed=42, save_plot_path=None):

    # Set seed for reproducibility
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    total_train_val = train_size + val_size
    train = df[target][-(total_train_val+forecast_window):-(val_size+forecast_window)].values
    val = df[target][-(val_size+forecast_window):-forecast_window].values
    test = df[target][-forecast_window:].values
    
    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train.reshape(-1, 1)).flatten()
    val_scaled = scaler.transform(val.reshape(-1, 1)).flatten()

    input_size = 1
    
    # Create Datasets
    train_dataset = TimeSeriesDataset(train_scaled, seq_length)
    use_pin_memory = torch.cuda.is_available()
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=use_pin_memory)
    
    val_dataset = TimeSeriesDataset(val_scaled, seq_length)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, pin_memory=use_pin_memory)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_losses = []
    val_losses = []
    
    best_val_loss = float('inf')
    
    # Create directory for models if it doesn't exist
    model_dir = f'best_models/seed_{seed}'
    os.makedirs(model_dir, exist_ok=True)
    best_model_path = f'{model_dir}/lstm_bs{batch_size}_hs{hidden_size}_dr{dropout}.pth'

    epochs_no_improve = 0
    best_epoch = 0

    # Training Time
    start_train_time = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            batch_x = batch_x.view(-1, seq_length, 1)

            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_train_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                batch_x = batch_x.view(-1, seq_length, 1) # Ensure shape
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / len(val_loader) if len(val_loader) > 0 else 0
        val_losses.append(avg_val_loss)
        
        # Save best model and Early Stopping
        if avg_val_loss <= best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            epochs_no_improve = 0
            best_epoch = epoch + 1
        else:
            epochs_no_improve += 1
            
        
            
    train_time = time.time() - start_train_time

    # Inference Time
    start_inference_time = time.time()
    
    # Load the best model
    model.load_state_dict(torch.load(best_model_path))
    print (f"Loaded best model in path: {best_model_path} with val loss: {best_val_loss:.6f} at epoch {best_epoch}")
    model.eval()
    forecast = []
    
    current_seq = val_scaled[-seq_length:].tolist()
    
    with torch.no_grad():
        for step in range(forecast_window):
            x = np.array(current_seq[-seq_length:]).reshape(-1, seq_length, 1) 
            x = torch.FloatTensor(x).to(device)
            pred = model(x).cpu().numpy()[0, 0]
            forecast.append(pred)
            current_seq.append(pred)
    
    forecast = scaler.inverse_transform(np.array(forecast).reshape(-1, 1)).flatten()
    
    inference_time = time.time() - start_inference_time
    
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)
    
    if save_plot_path:
        train_index = df[target][-(total_train_val+forecast_window):-(val_size+forecast_window)].index
        val_index = df[target][-(val_size+forecast_window):-forecast_window].index
        test_index = df[target][-forecast_window:].index
        
        plot_results(train, val, test, forecast, train_index, val_index, test_index, 
                     train_losses, val_losses, target, 
                     title=f'LSTM Forecast (Seed={seed}, BS={batch_size}, HS={hidden_size}, DO={dropout})',
                     save_path=save_plot_path)
    
    return rmse, mae, train_time, inference_time, best_epoch

# Grid Search Parameters
seeds = [42, 123, 2024]
batch_sizes = [4, 8, 16, 32]
hidden_sizes = [32, 64]
dropouts = [0.0, 0.1,0.2]

# Prepare results storage
results = []
os.makedirs('grid_search_plots', exist_ok=True)

# Run Grid Search
print(f"Starting Grid Search with {len(seeds) * len(batch_sizes) * len(hidden_sizes) * len(dropouts)} combinations...")

for seed in seeds:
    print(f"\n--- Processing Seed: {seed} ---")
    
    for batch_size, hidden_size, dropout in itertools.product(batch_sizes, hidden_sizes, dropouts):
        print(f"Running: Seed={seed}, BS={batch_size}, HS={hidden_size}, Dropout={dropout}")
        
        # Create directory for plots if it doesn't exist
        plot_dir = f'grid_search_plots/seed_{seed}'
        os.makedirs(plot_dir, exist_ok=True)
        plot_filename = f'{plot_dir}/lstm_bs{batch_size}_hs{hidden_size}_dr{dropout}.png'
        
        rmse, mae, train_time, infer_time, best_epoch = lstm_experiment_grid(
            df=df, 
            target=TARGET_COL, 
            train_size=train_size, 
            val_size=val_size,
            forecast_window=forecast_horizon, 
            seq_length=lookback_window,
            epochs=EPOCHS,  # Use the global EPOCHS setting (e.g., 1000)
            batch_size=batch_size, 
            lr=LEARNING_RATE,
            dropout=dropout,
            hidden_size=hidden_size,
            num_layers=1,
            patience=50,
            seed=seed,
            save_plot_path=plot_filename
        )
        
        results.append({
            'seed': seed,
            'batch_size': batch_size,
            'hidden_size': hidden_size,
            'dropout': dropout,
            'rmse': rmse,
            'mae': mae,
            'train_time': train_time,
            'inference_time': infer_time,
            'best_epoch': best_epoch,
            'plot_path': plot_filename
        })

# Convert to DataFrame
    results_df = pd.DataFrame(results)
    results_df.to_csv('grid_search_results.csv', index=False)

Starting Grid Search with 72 combinations...

--- Processing Seed: 42 ---
Running: Seed=42, BS=4, HS=32, Dropout=0.0
Loaded best model in path: best_models/seed_42/lstm_bs4_hs32_dr0.0.pth with val loss: 0.002121 at epoch 348
Running: Seed=42, BS=4, HS=32, Dropout=0.1


c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\.venv\Lib\site-packages\torch\nn\modules\rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Loaded best model in path: best_models/seed_42/lstm_bs4_hs32_dr0.1.pth with val loss: 0.002121 at epoch 348
Running: Seed=42, BS=4, HS=32, Dropout=0.2


c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\.venv\Lib\site-packages\torch\nn\modules\rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Loaded best model in path: best_models/seed_42/lstm_bs4_hs32_dr0.2.pth with val loss: 0.002121 at epoch 348
Running: Seed=42, BS=4, HS=64, Dropout=0.0
Loaded best model in path: best_models/seed_42/lstm_bs4_hs64_dr0.0.pth with val loss: 0.002141 at epoch 117
Running: Seed=42, BS=4, HS=64, Dropout=0.1


...

Loaded best model in path: best_models/seed_2024/lstm_bs32_hs32_dr0.2.pth with val loss: 0.001862 at epoch 745
Running: Seed=2024, BS=32, HS=64, Dropout=0.0
Loaded best model in path: best_models/seed_2024/lstm_bs32_hs64_dr0.0.pth with val loss: 0.001699 at epoch 535
Running: Seed=2024, BS=32, HS=64, Dropout=0.1


c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\.venv\Lib\site-packages\torch\nn\modules\rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Loaded best model in path: best_models/seed_2024/lstm_bs32_hs64_dr0.1.pth with val loss: 0.001699 at epoch 535
Running: Seed=2024, BS=32, HS=64, Dropout=0.2


c:\Users\andre.silva\OneDrive - Retail Consult\Desktop\Dissertation25-26\.venv\Lib\site-packages\torch\nn\modules\rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Loaded best model in path: best_models/seed_2024/lstm_bs32_hs64_dr0.2.pth with val loss: 0.001699 at epoch 535
